In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings("ignore")

#### Step 1: Create an imbalanced binary classification dataset

In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=2,
    n_redundant=8,
    weights=[0.9, 0.1],
    flip_y=0,
    random_state=42
)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

#### Split the dataset into training and testing sets

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

#### Experiment 1: Train Logistic Regression Classifier

In [4]:
log_reg = LogisticRegression(
    C=1,
    solver="liblinear"
)

log_reg.fit(X_train, y_train)

y_pred_log_reg = log_reg.predict(X_test)

print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



#### Experiment 2: Train Random Forest Classifier

In [5]:
rf_clf = RandomForestClassifier(
    n_estimators=30,
    max_depth=3
)

rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



#### Experiment 3: Train XGBoost

In [6]:
xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_clf.fit(X_train, y_train)

y_pred_xgb = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [7]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)

X_train_res, y_train_res = smt.fit_resample(
    X_train,
    y_train
)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [8]:
xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_clf.fit(X_train_res, y_train_res)

y_pred_xgb = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98       270
           1       0.78      0.83      0.81        30

    accuracy                           0.96       300
   macro avg       0.88      0.90      0.89       300
weighted avg       0.96      0.96      0.96       300



#### Track Experiments Using MLFlow

In [9]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(C=1, solver="liblinear"),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=30,
            max_depth=3
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBClassifier",
        XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss"
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "XGBClassifier With SMOTE",
        XGBClassifier(
            use_label_encoder=False,
            eval_metric="logloss"
        ),
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [ ]:
reports = []

for model_name, model, train_set, test_set in models:

    X_train = train_set[0]
    y_train = train_set[1]

    X_test = test_set[0]
    y_test = test_set[1]
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    reports.append(report)

In [11]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9675090252707581,
   'recall': 0.9925925925925926,
   'f1-score': 0.979890310786106,
   'support': 270.0},
  '1': {'precision': 0.9130434782608695,
   'recall': 0.7,
   'f1-score': 0.7924528301886793,
   'support': 30.0},
  'accuracy': 0.9633333333333334,
  'macro avg': {'precision': 0.9402762517658139,
   'recall': 0.8462962962962963,
   'f1-score': 0.8861715704873927,
   'support': 300.0},
  'weighted avg': {'precision': 0.9620

In [12]:
import mlflow

In [13]:
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")

for i,element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name',model_name)
        mlflow.log_metric('accuracy',report['accuracy'])
        mlflow.log_metric('recall_class_1',report['1']['recall'])
        mlflow.log_metric('recall_class_0',report['0']['recall'])
        mlflow.log_metric('f1_score_macro',report['macro avg']['f1-score'])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")
            

2026/07/04 12:32:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/04 12:32:00 INFO mlflow.store.db.utils: Updating database tables
2026/07/04 12:32:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/07/04 12:32:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/07/04 12:32:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/07/04 12:32:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/07/04 12:32:00 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.
2026/07/04 12:32:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/04 12:32:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/1a94b18da95d4ff28f7bb2e2f53312cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/07/04 12:32:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/59ffc3cffe8d4e3e8070ab7bddf83621
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/07/04 12:32:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/2/runs/220dd041ec8540488d04a53f62676802
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/2/runs/b77a98130aa34e39a1f10d6832baa8d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
